# NB63 — Z$_4$ Sector Algebra and Half-Integer Coupling

NB61 established the four $a_5$-dependent interference regimes (constructive,
destructive, mixed, complementary). NB62 mapped all 48 characters to SM
fermion quantum numbers and confirmed the 3+1 color theorem at level 1.

This notebook extracts the **algebraic spine** of the four sectors.

**Key discoveries**:
1. Every directed split decomposes as $S(a_5) = \text{Im}_1 + \text{Im}_2(a_5)$,
   where $\text{Im}_2$ follows a $Z_4$ cycle controlled by a **single** new
   parameter $\beta$ per $(a_3, a_7)$ pair.
2. $\beta = \text{Im}_2(a_5\!=\!1) \in \{\text{-}\tfrac{3}{2},\,\text{-}\tfrac{1}{2},\,
   +\tfrac{1}{2}\}$ — **half-integers**.
3. The product $S_1 \cdot S_3 = \text{Im}_1^2 - \beta^2$ is always **rational**,
   despite individual splits being irrational (lying in $\mathbb{Z}[\sqrt{3}]/2$).

**Two identities**:
- **#108**: Half-Integer Sector Coupling — $\beta \in \tfrac{1}{2}\mathbb{Z}$.
- **#109**: Rational Product Identity — $S_1\!\cdot\!S_3 \in \mathbb{Q}$.


In [1]:
# -- NB63 Setup --
import sys, os
_scripts = os.path.join(os.getcwd(), 'scripts')
if not os.path.isdir(_scripts):
    _scripts = os.path.join(os.getcwd(), '..', 'scripts')
sys.path.insert(0, _scripts)
import numpy as np
from fractions import Fraction
from solenoid_algebra import SA

all_indices = SA.all_character_indices()
N = SA.P        # 210
primes = SA.primes
phis = [SA.phi_p[p] for p in primes]
dlog = SA.dlog
s3 = np.sqrt(3)

ORDERS = {3: 2, 5: 4, 7: 6}
ACTIVE_PRIMES = [[3], [3, 7], [3, 5, 7]]
coupled_gens = [17, 23, 37]

def chi_val_level(idx, k, level):
    active = ACTIVE_PRIMES[level]
    phase = 0.0
    for p, phi_p, a in zip(primes, phis, idx):
        if phi_p == 1 or p not in active:
            continue
        r = k % p
        if r in dlog[p]:
            phase += 2 * np.pi * a * dlog[p][r] / phi_p
    return np.exp(1j * phase)

# Build inter-generation pairs (Gen1 <-> Gen2 only)
idx_to_pos = {idx: i for i, idx in enumerate(all_indices)}
conj_perm = []
for i, idx in enumerate(all_indices):
    a2, a3, a5, a7 = idx
    conj_idx = (a2, (-a3) % 2, (-a5) % 4, (-a7) % 6)
    conj_perm.append(idx_to_pos[conj_idx])

inter_pairs = []
visited = set()
for i, idx in enumerate(all_indices):
    if i in visited:
        continue
    j = conj_perm[i]
    if i == j:
        visited.add(i)
        continue
    gi, gj = idx[3] % 3, all_indices[j][3] % 3
    visited.add(i); visited.add(j)
    if gi == 1 and gj == 2:
        inter_pairs.append((i, j))
    elif gi == 2 and gj == 1:
        inter_pairs.append((j, i))

print(f'Group: Z*_{N}, |Z*| = {len(SA.Z_star)}')
print(f'Generators: {coupled_gens}')
print(f'Inter-generation pairs: {len(inter_pairs)}')
print(f'Tower levels: {["C_6", "C_42", "C_210"]}')

Group: Z*_210, |Z*| = 48
Generators: [17, 23, 37]
Inter-generation pairs: 16
Tower levels: ['C_6', 'C_42', 'C_210']


## 1. The $(Im_1, \beta)$ Parameterisation

Each inter-generation pair has a directed split at every $a_5$ sector.
Level 0 vanishes (NB61 #104). Level 1 is $a_5$-blind (only primes 3, 7 active).
So the split decomposes as:

$$S(a_5) = \underbrace{\text{Im}_1}_{a_5\text{-blind}} +\; \underbrace{\text{Im}_2(a_5)}_{a_5\text{-dependent}}$$

The $Z_4$ structure of $a_5 \in \{0,1,2,3\}$ produces four interference regimes,
all controlled by two numbers per $(a_3, a_7)$ pair:

| $a_5$ | $\text{Im}_2(a_5)$ | Regime |
|:---:|:---:|:---|
| 0 | $+\text{Im}_1$ | Constructive (doubled) |
| 1 | $\beta$ | Mixed |
| 2 | $-\text{Im}_1$ | Destructive (cancellation) |
| 3 | $-\beta$ | Complementary |

where $\beta \equiv \text{Im}_2(a_5\!=\!1)$ is the **sector coupling parameter**.


In [2]:
# -- Compute (Im1, beta) for all 4 (a3, a7) values --
sector_data = {}  # (a3, a7) -> {'im1': ..., 'beta': ..., 0: S0, 1: S1, ...}
for i1, i2 in inter_pairs:
    idx = all_indices[i1]
    a3, a5, a7 = idx[1], idx[2], idx[3]
    # Level contributions
    im1 = sum(chi_val_level(idx, g, 1).imag for g in coupled_gens)
    im2 = sum(chi_val_level(idx, g, 2).imag for g in coupled_gens)
    s_tower = im1 + im2
    key = (a3, a7)
    if key not in sector_data:
        sector_data[key] = {'im1': im1}
    sector_data[key][a5] = s_tower

# Extract beta = Im_2(a5=1) for each (a3, a7)
for key, data in sector_data.items():
    data['beta'] = data[1] - data['im1']  # S1 = Im1 + beta => beta = S1 - Im1

print('(a3,a7)   Im_1         beta        S0          S1          S2          S3')
print('-' * 90)
for (a3, a7), data in sorted(sector_data.items()):
    im1 = data['im1']
    beta = data['beta']
    color = 'LEPTON' if abs(abs(im1) - 3*s3/2) < 0.01 else 'quark'
    print(f'({a3},{a7}) [{color:>6}]  {im1:+8.4f}  {beta:+8.4f}  '
          f'{data[0]:+8.4f}  {data[1]:+8.4f}  {data[2]:+8.4f}  {data[3]:+8.4f}')

# Verify Z4 cycle: Im2(0)=Im1, Im2(1)=beta, Im2(2)=-Im1, Im2(3)=-beta
print()
for (a3, a7), data in sorted(sector_data.items()):
    im1, beta = data['im1'], data['beta']
    expected = {0: 2*im1, 1: im1+beta, 2: 0.0, 3: im1-beta}
    for a5 in range(4):
        assert abs(data[a5] - expected[a5]) < 1e-13, f'Z4 cycle FAIL at ({a3},{a7},a5={a5})'
print('PASS: Z4 cycle verified for all (a3, a7) pairs')


(a3,a7)   Im_1         beta        S0          S1          S2          S3
------------------------------------------------------------------------------------------
(0,1) [LEPTON]   +2.5981   +0.5000   +5.1962   +3.0981   -0.0000   +2.0981
(0,4) [ quark]   +0.8660   -0.5000   +1.7321   +0.3660   +0.0000   +1.3660
(1,1) [ quark]   -0.8660   -1.5000   -1.7321   -2.3660   -0.0000   +0.6340
(1,4) [ quark]   +0.8660   -0.5000   +1.7321   +0.3660   +0.0000   +1.3660

PASS: Z4 cycle verified for all (a3, a7) pairs


## 2. Half-Integer Sector Coupling (#108)

**Claim**: $\beta = \text{Im}_2(a_5\!=\!1)$ is always a **half-integer**:

$$\beta \in \left\{-\tfrac{3}{2},\; -\tfrac{1}{2},\; +\tfrac{1}{2}\right\}$$

with multiplicities $1 + 2 + 1 = 4$ across the four $(a_3, a_7)$ pairs.

This is non-trivial: each $\beta$ is a sum of three generators' imaginary parts
at the deepest tower level, yet all sums land exactly on half-integers.


In [3]:
# -- Verify beta in Z/2 (half-integers) --
print('Half-Integer Verification')
print('=' * 60)
beta_vals = []
for (a3, a7), data in sorted(sector_data.items()):
    beta = data['beta']
    beta_vals.append(beta)
    # Check that 2*beta is an integer
    twice = round(2 * beta)
    residual = abs(2 * beta - twice)
    status = 'PASS' if residual < 1e-13 else 'FAIL'
    frac = Fraction(twice, 2)
    print(f'  ({a3},{a7}): beta = {beta:+.4f} = {frac}   [{status}]')
    assert residual < 1e-13, f'beta not half-integer at ({a3},{a7})'

unique_betas = sorted(set(round(b, 10) for b in beta_vals))
print(f'\nDistinct beta values: {len(unique_betas)}')
for b in unique_betas:
    mult = sum(1 for v in beta_vals if abs(v - b) < 1e-10)
    frac = Fraction(round(2*b), 2)
    print(f'  beta = {frac} : multiplicity {mult}')

print(f'\n[PASS] Identity #108: beta in Z/2 = {{-3/2, -1/2, +1/2}}')
print(f'       Multiplicities: 1 + 2 + 1 = {len(beta_vals)}')


Half-Integer Verification
  (0,1): beta = +0.5000 = 1/2   [PASS]
  (0,4): beta = -0.5000 = -1/2   [PASS]
  (1,1): beta = -1.5000 = -3/2   [PASS]
  (1,4): beta = -0.5000 = -1/2   [PASS]

Distinct beta values: 3
  beta = -3/2 : multiplicity 1
  beta = -1/2 : multiplicity 2
  beta = 1/2 : multiplicity 1

[PASS] Identity #108: beta in Z/2 = {-3/2, -1/2, +1/2}
       Multiplicities: 1 + 2 + 1 = 4


## 3. Rational Product Identity (#109)

From the $Z_4$ decomposition:

$$S_1 \cdot S_3 = (\text{Im}_1 + \beta)(\text{Im}_1 - \beta) = \text{Im}_1^2 - \beta^2$$

Since $\text{Im}_1 \in \tfrac{\sqrt{3}}{2}\mathbb{Z}$ and $\beta \in \tfrac{1}{2}\mathbb{Z}$:
- $\text{Im}_1^2 \in \tfrac{3}{4}\mathbb{Z}$
- $\beta^2 \in \tfrac{1}{4}\mathbb{Z}$
- Their difference $\in \tfrac{1}{4}\mathbb{Z} \subset \mathbb{Q}$

The products of conjugate-sector splits are always **rational**, despite
individual splits being irrational.


In [4]:
# -- Verify S1*S3 = Im1^2 - beta^2 and rationality --
print('Rational Product Identity')
print('=' * 70)
print(f'{"(a3,a7)":>8} {"S1*S3":>10} {"Im1^2":>10} {"beta^2":>10} {"Im1^2-b^2":>10} {"Frac":>8}')
print('-' * 70)
all_rational = True
for (a3, a7), data in sorted(sector_data.items()):
    S1, S3 = data[1], data[3]
    im1, beta = data['im1'], data['beta']
    product = S1 * S3
    im1_sq = im1**2
    beta_sq = beta**2
    diff = im1_sq - beta_sq
    # Check rationality: 4*product should be an integer
    four_prod = round(4 * product)
    residual = abs(4 * product - four_prod)
    frac = Fraction(four_prod, 4)
    if residual > 1e-12:
        all_rational = False
    print(f'({a3},{a7})   {product:+10.4f} {im1_sq:10.4f} {beta_sq:10.4f} '
          f'{diff:+10.4f}  {frac}')
    assert abs(product - diff) < 1e-12, 'Product identity FAIL'

print()
assert all_rational, 'Rationality FAIL'
print('[PASS] Identity #109: S1*S3 = Im1^2 - beta^2')
print('       All products are rational (in Z/2)')

# Verify: sum of all products
total = sum(sector_data[k][1] * sector_data[k][3] for k in sector_data)
frac_total = Fraction(round(4*total), 4)
print(f'\n       Sum of S1*S3 over all (a3,a7): {frac_total} = {float(frac_total)}')


Rational Product Identity
 (a3,a7)      S1*S3      Im1^2     beta^2  Im1^2-b^2     Frac
----------------------------------------------------------------------
(0,1)      +6.5000     6.7500     0.2500    +6.5000  13/2
(0,4)      +0.5000     0.7500     0.2500    +0.5000  1/2
(1,1)      -1.5000     0.7500     2.2500    -1.5000  -3/2
(1,4)      +0.5000     0.7500     0.2500    +0.5000  1/2

[PASS] Identity #109: S1*S3 = Im1^2 - beta^2
       All products are rational (in Z/2)

       Sum of S1*S3 over all (a3,a7): 6 = 6.0


## 4. Complete Sector-Split Table

The full $4 \times 4$ table of directed splits $S(a_3, a_7; a_5)$, expressed
in exact algebraic form. Every entry is determined by just $(Im_1, \beta)$.


In [5]:
# -- Complete 4x4 table with exact algebraic expressions --
print('Complete Sector-Split Table')
print('=' * 80)
print()

# Express each S in terms of sqrt(3)
# S = Im1 + Im2, where Im1 in sqrt(3)/2 * Z and Im2 follows Z4 cycle
def algebraic_form(val):
    # Decompose val = a + b*sqrt(3) with a,b rational
    # Try: val = (p + q*sqrt(3))/2 for integer p, q
    for q in range(-6, 7):
        p = val - q * s3 / 2
        p_int = round(2 * p)
        if abs(2 * p - p_int) < 1e-10:
            return (Fraction(p_int, 2), Fraction(q, 2))
    return (None, None)

print(f'{"":>10} {"a5=0":>14} {"a5=1":>14} {"a5=2":>14} {"a5=3":>14}')
print('-' * 70)
for (a3, a7), data in sorted(sector_data.items()):
    row = f'({a3},{a7})    '
    for a5 in range(4):
        val = data[a5]
        p, q = algebraic_form(val)
        if q == 0:
            row += f'{"":>4}{p!s:>10}'
        else:
            if p == 0:
                row += f'{"":>2}{q!s}*s3{"":>6}'
            else:
                row += f' ({p!s}+{q!s}*s3)'
    print(row)

print()
print('where s3 = sqrt(3)')
print()

# Summary statistics
print('Algebraic structure:')
print(f'  Im_1 values: {{+/-sqrt(3)/2, +/-3*sqrt(3)/2}}')
print(f'  beta values: {{-3/2, -1/2, +1/2}}')
print(f'  All S in Z[sqrt(3)]/2    (ring of half-algebraic integers)')
print(f'  All S1*S3 in Q           (rational by product identity)')


Complete Sector-Split Table

                     a5=0           a5=1           a5=2           a5=3
----------------------------------------------------------------------
(0,1)      3*s3       (1/2+3/2*s3)             0 (-1/2+3/2*s3)
(0,4)      1*s3       (-1/2+1/2*s3)             0 (1/2+1/2*s3)
(1,1)      -1*s3       (-3/2+-1/2*s3)             0 (3/2+-1/2*s3)
(1,4)      1*s3       (-1/2+1/2*s3)             0 (1/2+1/2*s3)

where s3 = sqrt(3)

Algebraic structure:
  Im_1 values: {+/-sqrt(3)/2, +/-3*sqrt(3)/2}
  beta values: {-3/2, -1/2, +1/2}
  All S in Z[sqrt(3)]/2    (ring of half-algebraic integers)
  All S1*S3 in Q           (rational by product identity)


## 5. Color Persistence and Sector Splitting

At $a_5 = 0$: the 3-fold color degeneracy is **exact** (NB62 #106).
What happens at other sectors?

| $a_5$ | Multiplicity structure | Interpretation |
|:---:|:---|:---|
| 0 | 3 + 1 (exact) | Color-degenerate quarks + unique lepton |
| 1 | 2 + 1 + 1 | Partial color breaking: 2 quarks degenerate, 1 distinct |
| 2 | 4 (all zero) | Tower-protected: universal degeneracy |
| 3 | 1 + 2 + 1 | Mirror of $a_5=1$ (complementarity) |

The 3-fold degeneracy at $a_5=0$ **partially lifts** in the mixed sectors.
This structural difference between sectors is what allows NB61's
mass formula to distinguish quarks from leptons: $\log(m_\mu/m_e)/\log(m_s/m_d)
= 3(\rho+1)/(\rho+\sqrt{3})$ compares the lepton split at $a_5\!=\!0$ with
the quark split at $a_5\!=\!1$.


In [6]:
# -- Color structure by sector --
from collections import Counter
print('Multiplicity Structure by Sector')
print('=' * 60)
for a5_val in range(4):
    splits = []
    for (a3, a7), data in sorted(sector_data.items()):
        splits.append(round(abs(data[a5_val]), 10))
    counts = Counter(splits)
    mults = sorted(counts.items())
    mult_str = ' + '.join(f'{c}' for _, c in mults)
    print(f'\n  a5 = {a5_val}:  multiplicities [{mult_str}]')
    for val, cnt in mults:
        alg_p, alg_q = algebraic_form(val)
        if alg_q == 0:
            label = f'{alg_p}'
        elif alg_p == 0:
            label = f'{alg_q}*s3'
        else:
            label = f'|{alg_p}+{alg_q}*s3|'
        print(f'    |S| = {val:.4f} ({label}): mult = {cnt}')

# Verify a5=0 has exact 3+1
a5_0_splits = [round(abs(sector_data[k][0]), 10) for k in sector_data]
a5_0_counts = Counter(a5_0_splits)
assert len(a5_0_counts) == 2, 'a5=0 should have exactly 2 distinct |S| values'
mults = sorted(a5_0_counts.values())
assert mults == [1, 3], 'a5=0 should give 3+1 multiplicity'
print(f'\n[PASS] a5=0 has exact 3+1 color structure')

# Cross-sector mass formula verification
# NB61: lepton at (0,1) a5=0, quark at (1,1) a5=1
im1_l = sector_data[(0,1)]['im1']   # 3*sqrt(3)/2
im1_q = sector_data[(1,1)]['im1']   # -sqrt(3)/2
beta_q = sector_data[(1,1)]['beta'] # -3/2
print(f'\nNB61 cross-sector mass formula:')
print(f'  Lepton (0,1): Im1 = {im1_l:.4f} = 3*sqrt(3)/2')
print(f'  Quark  (1,1): Im1 = {im1_q:.4f} = -sqrt(3)/2, beta = {beta_q:.4f} = -3/2')
print(f'  S_eff(lepton) = (rho+1)*3*sqrt(3)/2')
print(f'  |S_eff(quark)| = (rho*sqrt(3) + 3)/2  [at a5=1]')
print(f'  Ratio = 3*(rho+1)/(rho+sqrt(3))  -- NB61 Identity #105')


Multiplicity Structure by Sector

  a5 = 0:  multiplicities [3 + 1]
    |S| = 1.7321 (1*s3): mult = 3
    |S| = 5.1962 (3*s3): mult = 1

  a5 = 1:  multiplicities [2 + 1 + 1]
    |S| = 0.3660 (|-1/2+1/2*s3|): mult = 2
    |S| = 2.3660 (|3/2+1/2*s3|): mult = 1
    |S| = 3.0981 (|1/2+3/2*s3|): mult = 1

  a5 = 2:  multiplicities [4]
    |S| = 0.0000 (0): mult = 4

  a5 = 3:  multiplicities [1 + 2 + 1]
    |S| = 0.6340 (|3/2+-1/2*s3|): mult = 1
    |S| = 1.3660 (|1/2+1/2*s3|): mult = 2
    |S| = 2.0981 (|-1/2+3/2*s3|): mult = 1

[PASS] a5=0 has exact 3+1 color structure

NB61 cross-sector mass formula:
  Lepton (0,1): Im1 = 2.5981 = 3*sqrt(3)/2
  Quark  (1,1): Im1 = -0.8660 = -sqrt(3)/2, beta = -1.5000 = -3/2
  S_eff(lepton) = (rho+1)*3*sqrt(3)/2
  |S_eff(quark)| = (rho*sqrt(3) + 3)/2  [at a5=1]
  Ratio = 3*(rho+1)/(rho+sqrt(3))  -- NB61 Identity #105


## 6. Scorecard


In [7]:
# -- NB63 Scorecard --
print('NB63 SCORECARD')
print('=' * 65)
print()
print('  #108  Half-Integer Sector Coupling          PASS')
print('        beta = Im_2(a5=1) in {-3/2, -1/2, +1/2}')
print('        All beta are half-integers (2*beta in Z)')
print()
print('  #109  Rational Product Identity              PASS')
print('        S1*S3 = Im1^2 - beta^2 in Q')
print('        Products of conjugate-sector splits rational')
print()
print('Running total: 109 predictions/identities, 0 free parameters')
print()
print('KEY STRUCTURAL RESULT:')
print('All 64 directed splits (16 pairs x 4 sectors) are determined')
print('by a 4-row table of (Im_1, beta) values, with:')
print('  Im_1 in sqrt(3)/2 * Z    (irrational)')
print('  beta in Z/2              (rational)')
print('  S in Z[sqrt(3)]/2        (algebraic)')
print('  S1*S3 in Z/2             (rational)')


NB63 SCORECARD

  #108  Half-Integer Sector Coupling          PASS
        beta = Im_2(a5=1) in {-3/2, -1/2, +1/2}
        All beta are half-integers (2*beta in Z)

  #109  Rational Product Identity              PASS
        S1*S3 = Im1^2 - beta^2 in Q
        Products of conjugate-sector splits rational

Running total: 109 predictions/identities, 0 free parameters

KEY STRUCTURAL RESULT:
All 64 directed splits (16 pairs x 4 sectors) are determined
by a 4-row table of (Im_1, beta) values, with:
  Im_1 in sqrt(3)/2 * Z    (irrational)
  beta in Z/2              (rational)
  S in Z[sqrt(3)]/2        (algebraic)
  S1*S3 in Z/2             (rational)
